In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


In [3]:
# Example sentences (you can replace this with your own corpus)
texts = [
    "Hello, how are you?",
    "I am fine, thank you!",
    "How about you?",
    "I'm doing great, thanks for asking!"
]

# Tokenize the text into integers
tokenizer = Tokenizer()
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)

# Pad sequences to ensure they are of the same length
max_length = max([len(seq) for seq in sequences])
padded_sequences = pad_sequences(sequences, maxlen=max_length, padding='post')

# Vocabulary size (number of unique words)
vocab_size = len(tokenizer.word_index) + 1  # Adding 1 for the padding token


In [5]:
def build_autoencoder(vocab_size, max_length):
    input_seq = layers.Input(shape=(max_length,))
    
    # Encoder
    x = layers.Embedding(vocab_size, 64, input_length=max_length)(input_seq)
    x = layers.LSTM(128, activation='relu')(x)
    
    # Latent space (compressed representation)
    latent = layers.Dense(64, activation='relu')(x)
    
    # Decoder
    x = layers.RepeatVector(max_length)(latent)  # Repeat the latent vector to match the input length
    x = layers.LSTM(128, activation='relu', return_sequences=True)(x)
    decoded = layers.Dense(vocab_size, activation='softmax')(x)
    
    # Define the model
    model = models.Model(input_seq, decoded)
    
    # Compile the model
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    
    return model


In [7]:
# Prepare the input data (X) and the output data (Y) for training
X_train = padded_sequences
Y_train = np.expand_dims(padded_sequences, -1)  # Adding a channel dimension for sparse categorical crossentropy

# Build the autoencoder model
model = build_autoencoder(vocab_size, max_length)

# Train the model
model.fit(X_train, Y_train, epochs=50, batch_size=32)


Epoch 1/50


C:\Users\TK\anaconda3\Lib\site-packages\keras\src\layers\core\embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step - accuracy: 0.0417 - loss: 2.7726
Epoch 2/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.2500 - loss: 2.7697
Epoch 3/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.2500 - loss: 2.7669
Epoch 4/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.2500 - loss: 2.7638
Epoch 5/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.2500 - loss: 2.7604
Epoch 6/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.2500 - loss: 2.7564
Epoch 7/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.2500 - loss: 2.7517
Epoch 8/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.2500 - loss: 2.7462
Epoch 9/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - accuracy: 0.2500 - loss: 2.7397
Epoch 10/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.2500 - loss: 2.7318
Epoch 11/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.2500 - loss: 2.7221
Epoch 12/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.2500 - loss: 2.7104
Epoch 13/50
1/

In [11]:
def generate_text(model, tokenizer, seed_text, max_length, n_words):
    # Tokenize the seed text
    seed_seq = tokenizer.texts_to_sequences([seed_text])
    seed_seq = pad_sequences(seed_seq, maxlen=max_length, padding='post')

    # Generate words one by one
    generated_text = seed_text
    for _ in range(n_words):
        # Predict the next word
        predicted_probs = model.predict(seed_seq, verbose=0)
        predicted_token = np.argmax(predicted_probs[0, -1, :])  # Take the word with the highest probability
        
        # Skip padding token (0)
        if predicted_token == 0:
            continue
        
        # Convert token to word
        predicted_word = tokenizer.index_word.get(predicted_token, "<unk>")  # Use "<unk>" for unknown tokens
        
        # Append predicted word to the generated text
        generated_text += ' ' + predicted_word
        
        # Update seed sequence with the predicted word
        seed_seq = np.roll(seed_seq, shift=-1, axis=1)
        seed_seq[0, -1] = predicted_token
    
    return generated_text


In [13]:
# Generate new text based on the seed
seed_text = "How are you"
generated = generate_text(model, tokenizer, seed_text, max_length, n_words=5)
print(generated)


How are you
